In [1]:
import geopandas as gpd
from utils.rsei_utils import aggregate_RSEI

In [2]:
# Configurations

datadir = '/projects/standard/lenkne/oboiko/EJ/data/'
huc12_filepath = datadir + 'aoi_huc12_boundaries.gpkg'
out_filepath = datadir + 'huc12_rsei_toxconc_weighted.csv'
study_periods = ['2008_2012', '2013_2017', '2018_2022']
target_crs = 'EPSG:5070'

In [3]:
# Load study area
huc12 = gpd.read_file(huc12_filepath)
huc12 = huc12.to_crs(target_crs)
huc12.head()

ERROR 1: PROJ: proj_create_from_database: Open of /users/2/oboiko/.conda/envs/geo/share/proj failed


,huc12,name,areasqkm,states,Region,geometry
0,070300050401,Forest Lake-Sunrise River,43.43,MN,Gorge,"MULTIPOLYGON (((240228.337 2483439.302, 240271..."
1,070900070402,Fairfield Ditch Number 1-Green River,90.16,IL,Working River,"MULTIPOLYGON (((517568.047 2070542.219, 517539..."
2,070300030103,West Branch Kettle River,101.59,MN,Headwaters,"MULTIPOLYGON (((234614.153 2633575.283, 234693..."
3,070400080902,Crystal Creek,41.77,MN,Driftless,"MULTIPOLYGON (((362547.97 2317024.237, 362554...."
4,070400030605,Rose Valley,39.09,WI,Driftless,"MULTIPOLYGON (((332805.751 2374160.973, 332815..."


In [4]:
%%time
for study_period in study_periods:
    print ('\nProcessing', study_period)
    years = list(range(int(study_period.split('_')[0]), int(study_period.split('_')[1]) + 1))
    print (years)
    # aggregate rsei to 5-year period
    # this sums up on-site and off-site and averages over years in the study period
    print ('Aggregate RSEI flowlines to study period and to HUC12 units')
    flowlines = aggregate_RSEI(years, selection_aoi=huc12,  universe='core01')
    # Ensure both GDFs are in the same projected CRS (EPSG:5070)
    # This is critical for accurate length calculations in meters
    flowlines = flowlines.to_crs(target_crs)
    # Intersect the two layers
    # This 'cuts' the flowlines at the polygon boundaries and 
    # attaches the polygon's unique ID (e.g., 'huc12') to each line segment.
    intersected = gpd.overlay(flowlines, huc12, how='intersection')
    # Calculate the length of the clipped river segments
    intersected['length'] = intersected.geometry.length
    # Calculate weighted concentrations (Concentration * Segment Length)
    intersected['TOXCONC_weighed'] = intersected['TOXCONC'] * intersected['length']
    # Aggregate the data by the huc12 id
    # We need the sum of the weighted values and the sum of lengths for each polygon.
    grouped = intersected.groupby('huc12').agg({
        'TOXCONC_weighed': 'sum',
        'length': 'sum'
    })
    # Calculate the final length-weighted average
    grouped[f'{study_period}_TOXCONC'] = grouped['TOXCONC_weighed'] / grouped['length']
    # Bring the results back to your original Polygons GeoDataFram
    huc12[f'{study_period}_TOXCONC'] = huc12['huc12'].map(grouped[f'{study_period}_TOXCONC'])


Processing 2008_2012
[2008, 2009, 2010, 2011, 2012]
Aggregate RSEI flowlines to study period and to HUC12 units

Processing 2013_2017
[2013, 2014, 2015, 2016, 2017]
Aggregate RSEI flowlines to study period and to HUC12 units

Processing 2018_2022
[2018, 2019, 2020, 2021, 2022]
Aggregate RSEI flowlines to study period and to HUC12 units
CPU times: user 1min 16s, sys: 2.26 s, total: 1min 18s
Wall time: 1min 21s


In [5]:
# save results to a tabular format
# geometries can be dropped (will join to the boundaries file as needed)
huc12.drop(columns=['geometry']).to_csv(out_filepath)